In [1]:
# import polars as pl

# df = pl.read_parquet(r'/home/d.golomolzin/kiber-polka/meta_data.parquet')
# df = df.drop('target_2_6')
# df.write_parquet('meta_data.parquet')

In [2]:
# import polars as pl

# df = pl.read_parquet(r'/home/d.golomolzin/kiber-polka/meta_data.parquet')
# df

In [3]:
import os
import json

import polars as pl
from catboost import CatBoostClassifier, Pool


snapshots_path = r'snapshots'
snapshots = sorted(
    [os.path.join(snapshots_path, snapshot) for snapshot in os.listdir(snapshots_path)], key=lambda x: (len(x), x))
configs_path = r'configs'
configs   = sorted(
    [os.path.join(configs_path, config) for config in os.listdir(configs_path)], key=lambda x: (len(x), x))

df_test = pl.read_parquet(r'data/test_main_features_typed.parquet')
idxs = df_test['customer_id']
X_test = df_test.drop('customer_id')

In [4]:
configs

['configs/target_1_1.json',
 'configs/target_1_2.json',
 'configs/target_1_3.json',
 'configs/target_1_4.json',
 'configs/target_1_5.json',
 'configs/target_2_1.json',
 'configs/target_2_2.json',
 'configs/target_2_3.json',
 'configs/target_2_4.json',
 'configs/target_2_5.json',
 'configs/target_2_6.json',
 'configs/target_2_7.json',
 'configs/target_2_8.json',
 'configs/target_3_1.json',
 'configs/target_3_2.json',
 'configs/target_3_3.json',
 'configs/target_3_4.json',
 'configs/target_3_5.json',
 'configs/target_4_1.json',
 'configs/target_5_1.json',
 'configs/target_5_2.json',
 'configs/target_6_1.json',
 'configs/target_6_2.json',
 'configs/target_6_3.json',
 'configs/target_6_4.json',
 'configs/target_6_5.json',
 'configs/target_7_1.json',
 'configs/target_7_2.json',
 'configs/target_7_3.json',
 'configs/target_8_1.json',
 'configs/target_8_2.json',
 'configs/target_8_3.json',
 'configs/target_9_1.json',
 'configs/target_9_2.json',
 'configs/target_9_3.json',
 'configs/target_9_4

In [5]:
predicts = {}
predicts['customer_id'] = idxs
for ind_target in range(41):

    cbm_path    = snapshots[ind_target]
    config_path = configs[ind_target]

    with open(config_path, 'r', encoding='utf-8') as file:
        config = json.load(file)

    model = CatBoostClassifier().load_model(cbm_path)

    required_features = model.feature_names_
    sub_X_test = X_test.select(required_features)

    X_test_pl = Pool(
        data=sub_X_test, 
        cat_features=[feature for feature in config['category'] if feature in required_features],
    )
    y_pred = model.predict_proba(X_test_pl)[:,1]

    predicts[config['target'].replace('target', 'predict', 1)] = y_pred

submit = pl.DataFrame(predicts)
submit.write_parquet('submit_01.parquet')

In [6]:
d = pl.read_parquet('submit_01.parquet')
d

customer_id,predict_1_1,predict_1_2,predict_1_3,predict_1_4,predict_1_5,predict_2_1,predict_2_2,predict_2_3,predict_2_4,predict_2_5,predict_2_6,predict_2_7,predict_2_8,predict_3_1,predict_3_2,predict_3_3,predict_3_4,predict_3_5,predict_4_1,predict_5_1,predict_5_2,predict_6_1,predict_6_2,predict_6_3,predict_6_4,predict_6_5,predict_7_1,predict_7_2,predict_7_3,predict_8_1,predict_8_2,predict_8_3,predict_9_1,predict_9_2,predict_9_3,predict_9_4,predict_9_5,predict_9_6,predict_9_7,predict_9_8,predict_10_1
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,0.000638,0.001598,0.009127,0.012016,0.001051,0.023939,0.004379,0.000548,0.006821,0.00102,0.00285,0.00007,0.000002,0.11597,0.018002,0.000486,0.000231,0.000005,0.017307,0.004139,0.001455,0.002271,0.002274,0.003593,0.000922,0.000053,0.064208,0.014883,0.004948,0.001242,0.007372,0.025349,0.003948,0.065696,0.020327,0.000144,0.002555,0.372243,0.120422,0.002068,0.324144
1750002,0.008744,0.002297,0.047046,0.019958,0.002395,0.034314,0.010797,0.000157,0.018941,0.001733,0.002441,0.000181,8.1427e-7,0.056097,0.006966,0.002782,0.000046,0.000014,0.001591,0.004087,0.001015,0.004053,0.004514,0.002136,0.001519,0.000002,0.047136,0.014186,0.001244,0.007239,0.046568,0.007344,0.001178,0.195957,0.025139,0.018067,0.018433,0.376068,0.108416,0.000222,0.286678
1750003,0.002539,0.004935,0.01071,0.036948,0.001305,0.007839,0.003274,0.000729,0.002488,0.001825,0.005801,0.000024,0.000027,0.141201,0.036284,0.000218,0.000266,0.000007,0.02319,0.005266,0.000312,0.006001,0.002965,0.00231,0.003101,0.000042,0.003098,0.01694,0.004203,0.045699,0.024078,0.011547,0.00429,0.04212,0.012466,0.00007,0.00094,0.194164,0.061853,0.001928,0.336926
1750004,0.000136,0.000617,0.002451,0.005796,0.000126,0.008514,0.001632,0.000235,0.007614,0.000328,0.001518,0.000051,0.000004,0.060967,0.011804,0.000662,0.000024,0.000005,0.00208,0.002163,0.000619,0.002523,0.003663,0.003558,0.000777,0.000018,0.053851,0.011161,0.006945,0.003295,0.003872,0.016858,0.003031,0.038164,0.02137,0.000149,0.003459,0.361655,0.135696,0.004437,0.321799
1750005,0.010492,0.002067,0.021243,0.022564,0.000325,0.003605,0.000811,0.000552,0.001496,0.002214,0.002976,0.00009,0.000003,0.144702,0.008189,0.000515,0.000054,0.000001,0.001143,0.001646,0.000815,0.004788,0.00707,0.000871,0.000883,0.000004,0.000419,0.00946,0.001027,0.005266,0.035113,0.00653,0.002883,0.058727,0.025119,0.000238,0.008508,0.230532,0.112185,0.000673,0.31957
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1999996,0.002629,0.000775,0.002688,0.007107,0.000469,0.000237,0.001035,0.00008,0.00326,0.000553,0.004335,0.000006,8.2217e-7,0.048049,0.011147,0.001315,0.000062,0.000002,0.00086,0.002421,0.000938,0.006796,0.007216,0.004366,0.001839,0.000034,0.043503,0.012753,0.001355,0.002756,0.028161,0.002249,0.00023,0.001497,0.013136,0.003268,0.001435,0.111951,0.00638,0.000381,0.728206
1999997,0.001605,0.001744,0.007503,0.006704,0.001076,0.012099,0.001434,0.019359,0.002839,0.000602,0.001631,0.00047,0.000005,0.220471,0.442121,0.000674,0.012421,0.000682,0.011279,0.024236,0.002381,0.004581,0.006521,0.008256,0.005216,0.00021,0.118177,0.013151,0.002835,0.015112,0.020732,0.002953,0.001115,0.02234,0.007794,0.000283,0.000781,0.086579,0.049401,0.01453,0.272837
1999998,0.000395,0.000133,0.000446,0.001135,0.000091,0.000471,0.000082,0.000616,0.004,0.001179,0.003733,0.000007,5.0321e-8,0.0717,0.437649,0.000031,0.000789,0.011557,0.002128,0.005172,0.00077,0.013992,0.000553,0.003263,0.024163,0.000159,0.00956,0.001172,0.000629,0.85393,0.002542,0.003505,0.000569,0.00208,0.005776,0.00001,0.000161,0.014305,0.005652,0.000517,0.070203
